# Lesson 4 — LangGraph: your proxy's logic as a graph

**Goal:** rebuild the heart of your proxy — load history, call the model, save
the turn — as a LangGraph graph of nodes and edges.

## Why a graph at all?

A LangChain chain (Lesson 2) is a straight line `A | B | C`. Your proxy does
bookkeeping *around* the model call (read history first, save the turn after),
and real apps branch ("did the model ask for a tool? loop"). A graph expresses
that explicitly.

### The proxy parallel — `api/proxy/main.py:231-327`
Your `chat_completions` handler runs load → inject → call → save as one long
function where order is just top-to-bottom code. Below, each step is a named
**node** and the order is **edges** you can see and rewire.

## The three pieces of any LangGraph
1. **State** — a shared dict flowing through the graph. Each node reads it and
   returns updates. Here it holds the conversation `messages`.
2. **Nodes** — plain functions `state -> dict`.
3. **Edges** — wiring: which node runs after which.

We use an in-memory list as a stand-in for your SQLite history so the notebook
runs on its own.

In [ ]:
from typing import Annotated, TypedDict
from langchain_openai import ChatOpenAI
from langchain_core.messages import AnyMessage, HumanMessage
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages

# Fake 'database' — in your app this is SQLite (api/proxy/database.py)
FAKE_HISTORY: list[dict] = []

### The state
`Annotated[list, add_messages]` tells the graph: when a node returns new
`messages`, **append** them to the existing list rather than overwrite.
`add_messages` is the built-in reducer for that.

In [ ]:
class ChatState(TypedDict):
    messages: Annotated[list[AnyMessage], add_messages]

model = ChatOpenAI(base_url='http://127.0.0.1:8080/v1', api_key='not-needed',
                   model='qwen3-14b-q4_k_m', temperature=0.7)

### The three nodes
Each mirrors a piece of your proxy: `load_history` ≈ `get_recent_messages`,
`call_model` ≈ the httpx POST, `save_turn` ≈ `save_turn_background`.

In [ ]:
def load_history(state: ChatState) -> dict:
    prior = [HumanMessage(content=h['content']) if h['role'] == 'user'
             else ('ai', h['content']) for h in FAKE_HISTORY]
    print(f'[load_history] pulled {len(prior)} prior message(s)')
    return {'messages': prior}

def call_model(state: ChatState) -> dict:
    print(f"[call_model] sending {len(state['messages'])} message(s) to the model")
    reply = model.invoke(state['messages'])
    return {'messages': [reply]}

def save_turn(state: ChatState) -> dict:
    last_user = next((m for m in reversed(state['messages']) if m.type == 'human'), None)
    last_ai = state['messages'][-1]
    if last_user:
        FAKE_HISTORY.append({'role': 'user', 'content': last_user.content})
    FAKE_HISTORY.append({'role': 'assistant', 'content': last_ai.content})
    print(f'[save_turn] db now has {len(FAKE_HISTORY)} rows')
    return {}

### Wire the graph
`START` and `END` are built-in entry/exit markers. `.compile()` turns the
blueprint into a Runnable — so the graph has `.invoke()` just like a chain.

In [ ]:
builder = StateGraph(ChatState)
builder.add_node('load_history', load_history)
builder.add_node('call_model', call_model)
builder.add_node('save_turn', save_turn)

builder.add_edge(START, 'load_history')
builder.add_edge('load_history', 'call_model')
builder.add_edge('call_model', 'save_turn')
builder.add_edge('save_turn', END)

graph = builder.compile()
print(graph.get_graph().draw_ascii())

### Run it twice to watch memory accumulate across 'requests'
The first call starts with empty history; the second sees the first turn.

In [ ]:
out = graph.invoke({'messages': [HumanMessage(content='My name is Kunal.')]})
print('MODEL SAID:', out['messages'][-1].content)

In [ ]:
out = graph.invoke({'messages': [HumanMessage(content="What's my name?")]})
print('MODEL SAID:', out['messages'][-1].content)

---
**The payoff:** changing this flow is now editing the four `add_edge` lines —
not re-reading a 100-line handler. Adding "if the model wants a tool, loop
back" is one *conditional* edge. That branch is the whole reason LangGraph
exists, and it's where Lesson 5 goes.